In [17]:
import time, random, requests
import pandas as pd

API_KEY = "YOUR_API_KEY_HERE"
ACCESS_LEVEL = "trial"     # "trial" or "production"
LANG = "en"
FORMAT = "json"

def sr_get(url, params, timeout=30, max_retries=8, base_sleep=1.0):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=timeout)

        if r.status_code == 200:
            return r

        if r.status_code in (429, 500, 502, 503, 504):
            retry_after = r.headers.get("Retry-After")
            sleep_s = float(retry_after) if retry_after else base_sleep * (2 ** attempt)
            sleep_s += random.uniform(0, 0.5)
            print(f"HTTP {r.status_code} → retry {attempt+1}/{max_retries} in {sleep_s:.2f}s")
            time.sleep(sleep_s)
            continue

        r.raise_for_status()

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")

# 1) Get live match list
summ_url = f"https://api.sportradar.com/darts/{ACCESS_LEVEL}/v2/{LANG}/schedules/live/summaries.{FORMAT}"
live = sr_get(summ_url, params={"api_key": API_KEY}).json()

events = live.get("sport_events") or live.get("summaries") or []
if not events:
    raise RuntimeError("No live events returned. Try daily summaries instead.")

# pick first event
se = events[0].get("sport_event", events[0])
SPORT_EVENT_ID = se["id"]

print("Using match:", SPORT_EVENT_ID,
      "| competition:", se.get("sport_event_context", {}).get("competition", {}).get("name"),
      "| start:", se.get("start_time"))

# 2) Fetch match timeline (dart-by-dart)
tl_url = f"https://api.sportradar.com/darts/{ACCESS_LEVEL}/v2/{LANG}/sport_events/{SPORT_EVENT_ID}/timeline.{FORMAT}"
timeline = sr_get(tl_url, params={"api_key": API_KEY}).json()

# 3) Extract dart events
def seg(score, mult):
    if score is None or mult is None:
        return None
    if score == 25 and mult == 1: return "SB"
    if score == 25 and mult == 2: return "DB"
    if mult == 1: return f"S{score}"
    if mult == 2: return f"D{score}"
    if mult == 3: return f"T{score}"
    return f"{mult}x{score}"

rows = []
for ev in timeline.get("timeline", []):
    if ev.get("type") == "dart":
        score = ev.get("dart_score")
        mult  = ev.get("dart_score_multiplier")
        total = ev.get("dart_score_total")
        rows.append({
            "event_id": ev.get("id"),
            "time": ev.get("time"),
            "competitor": ev.get("competitor"),  # home/away
            "dart_score": score,
            "multiplier": mult,
            "dart_total": total,
            "segment": seg(score, mult),
            "is_bust": ev.get("is_bust"),
            "is_checkout_attempt": ev.get("is_checkout_attempt"),
            "is_gameshot": ev.get("is_gameshot"),
            "home_score": ev.get("home_score"),
            "away_score": ev.get("away_score"),
            "period": ev.get("period"),
        })

df_darts = pd.DataFrame(rows).sort_values(["time", "event_id"])
df_darts.head(200)


Using match: sr:sport_event:68640096 | competition: Modus Super Series | start: 2026-02-06T16:45:00+00:00


,event_id,time,competitor,dart_score,multiplier,dart_total,segment,is_bust,is_checkout_attempt,is_gameshot,home_score,away_score,period
0,2263348266,2026-02-06T16:46:44+00:00,home,20,1,20,S20,False,False,None,None,None,None
1,2263348284,2026-02-06T16:46:46+00:00,home,20,1,20,S20,False,False,None,None,None,None
2,2263348306,2026-02-06T16:46:48+00:00,home,20,1,20,S20,False,False,None,None,None,None
3,2263348382,2026-02-06T16:46:54+00:00,away,20,1,20,S20,False,False,None,None,None,None
4,2263348406,2026-02-06T16:46:56+00:00,away,20,1,20,S20,False,False,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2263357054,2026-02-06T16:59:36+00:00,home,19,1,19,S19,False,False,None,None,None,None
196,2263357082,2026-02-06T16:59:38+00:00,home,19,1,19,S19,False,False,None,None,None,None
197,2263357158,2026-02-06T16:59:45+00:00,away,20,1,20,S20,False,False,None,None,None,None
198,2263357168,2026-02-06T16:59:47+00:00,away,20,1,20,S20,False,False,None,None,None,None


In [18]:
dart_ev = next(ev for ev in timeline["timeline"] if ev.get("type") == "dart")
dart_ev.keys()


dict_keys(['id', 'type', 'time', 'competitor', 'dart_score', 'dart_score_multiplier', 'dart_score_total', 'is_checkout_attempt', 'is_bust'])

In [19]:
non_dart_ev = next(ev for ev in timeline["timeline"] if ev.get("type") != "dart")
non_dart_ev.get("type"), non_dart_ev.keys()


('match_info',
 dict_keys(['id', 'type', 'time', 'best_of_sets', 'best_of_legs']))

In [21]:
import time, random, requests
import pandas as pd
from collections import Counter

API_KEY = "YOUR_API_KEY_HERE"
ACCESS_LEVEL = "trial"     # "trial" or "production"
LANG = "en"
FORMAT = "json"

def sr_get(url, params, timeout=30, max_retries=8, base_sleep=1.0):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=timeout)
        if r.status_code == 200:
            return r
        if r.status_code in (429, 500, 502, 503, 504):
            retry_after = r.headers.get("Retry-After")
            sleep_s = float(retry_after) if retry_after else base_sleep * (2 ** attempt)
            sleep_s += random.uniform(0, 0.5)
            print(f"HTTP {r.status_code} → retry {attempt+1}/{max_retries} in {sleep_s:.2f}s")
            time.sleep(sleep_s)
            continue
        r.raise_for_status()
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")

def seg(score, mult):
    if score is None or mult is None:
        return None
    if score == 25 and mult == 1: return "SB"
    if score == 25 and mult == 2: return "DB"
    if mult == 1: return f"S{score}"
    if mult == 2: return f"D{score}"
    if mult == 3: return f"T{score}"
    return f"{mult}x{score}"

# 1) Get live match list
summ_url = f"https://api.sportradar.com/darts/{ACCESS_LEVEL}/v2/{LANG}/schedules/live/summaries.{FORMAT}"
live = sr_get(summ_url, params={"api_key": API_KEY}).json()

events = live.get("sport_events") or live.get("summaries") or []
if not events:
    raise RuntimeError("No live events returned. Try daily summaries instead.")

se = events[0].get("sport_event", events[0])
SPORT_EVENT_ID = se["id"]

print("Using match:", SPORT_EVENT_ID,
      "| competition:", se.get("sport_event_context", {}).get("competition", {}).get("name"),
      "| start:", se.get("start_time"))

# 2) Fetch match timeline
tl_url = f"https://api.sportradar.com/darts/{ACCESS_LEVEL}/v2/{LANG}/sport_events/{SPORT_EVENT_ID}/timeline.{FORMAT}"
timeline = sr_get(tl_url, params={"api_key": API_KEY}).json()

timeline_events = timeline.get("timeline", [])
print("Event types:", Counter([e.get("type") for e in timeline_events]))

# 3) Build dart-by-dart with computed state
START_LEG_SCORE = 501
home_rem = START_LEG_SCORE
away_rem = START_LEG_SCORE

current_thrower = None
dart_in_visit = 0
home_visit_start = home_rem
away_visit_start = away_rem

rows = []

for ev in sorted(timeline_events, key=lambda x: (x.get("time",""), x.get("id",0))):
    et = ev.get("type")

    # Reset leg if timeline contains explicit leg/period start signals (best-effort)
    if et and ("period_start" in et or "leg_start" in et or "new_leg" in et):
        home_rem = START_LEG_SCORE
        away_rem = START_LEG_SCORE
        current_thrower = None
        dart_in_visit = 0
        home_visit_start = home_rem
        away_visit_start = away_rem

    if et != "dart":
        continue

    comp = ev.get("competitor")  # "home"/"away"
    score = ev.get("dart_score")
    mult  = ev.get("dart_score_multiplier")
    total = ev.get("dart_score_total")
    bust  = ev.get("is_bust")

    # New visit when thrower changes
    if comp != current_thrower:
        current_thrower = comp
        dart_in_visit = 0
        home_visit_start = home_rem
        away_visit_start = away_rem

    dart_in_visit += 1

    rem_before = home_rem if comp == "home" else away_rem
    opp_before = away_rem if comp == "home" else home_rem

    if total is None:
        total = (score or 0) * (mult or 1)

    rem_after = rem_before if bust else rem_before - total

    # Update running remaining
    if comp == "home":
        home_rem = home_visit_start if bust else rem_after
    else:
        away_rem = away_visit_start if bust else rem_after

    rows.append({
        "event_id": ev.get("id"),
        "time": ev.get("time"),
        "competitor": comp,
        "dart_in_visit": dart_in_visit,
        "score_remaining_before": rem_before,
        "opp_remaining_before": opp_before,
        "dart_score": score,
        "multiplier": mult,
        "dart_total": total,
        "segment": seg(score, mult),
        "is_bust": bust,
        "is_checkout_attempt": ev.get("is_checkout_attempt"),
        "is_gameshot": ev.get("is_gameshot"),
    })

df_darts2 = pd.DataFrame(rows).sort_values(["time","event_id"]).reset_index(drop=True)
df_darts2.head(50)


Using match: sr:sport_event:68640098 | competition: Modus Super Series | start: 2026-02-06T17:05:00+00:00
Event types: Counter({'dart': 208, 'score_change': 65, 'leg_score_change': 6, 'match_info': 1, 'first_throw': 1, 'match_started': 1, 'period_start': 1, 'match_ended': 1})


,event_id,time,competitor,dart_in_visit,score_remaining_before,opp_remaining_before,dart_score,multiplier,dart_total,segment,is_bust,is_checkout_attempt,is_gameshot
0,2263363342,2026-02-06T17:06:17+00:00,home,1,501,501,20,1,20,S20,False,False,None
1,2263363368,2026-02-06T17:06:20+00:00,home,2,481,501,20,3,60,T20,False,False,None
2,2263363378,2026-02-06T17:06:21+00:00,home,3,421,501,5,1,5,S5,False,False,None
3,2263363440,2026-02-06T17:06:24+00:00,away,1,501,416,20,1,20,S20,False,False,None
4,2263363474,2026-02-06T17:06:26+00:00,away,2,481,416,20,1,20,S20,False,False,None
5,2263363538,2026-02-06T17:06:29+00:00,away,3,461,416,17,3,51,T17,False,False,None
6,2263363606,2026-02-06T17:06:33+00:00,home,1,416,410,20,1,20,S20,False,False,None
7,2263363644,2026-02-06T17:06:35+00:00,home,2,396,410,19,1,19,S19,False,False,None
8,2263363662,2026-02-06T17:06:36+00:00,home,3,377,410,19,1,19,S19,False,False,None
9,2263363716,2026-02-06T17:06:40+00:00,away,1,410,358,20,1,20,S20,False,False,None


In [22]:
Counter([e.get("type") for e in timeline["timeline"]])


Counter({'dart': 208,
         'score_change': 65,
         'leg_score_change': 6,
         'match_info': 1,
         'first_throw': 1,
         'match_started': 1,
         'period_start': 1,
         'match_ended': 1})

In [24]:
import pandas as pd

timeline_events = sorted(timeline["timeline"], key=lambda x: (x.get("time",""), x.get("id",0)))

def seg(score, mult):
    if score is None or mult is None:
        return None
    if score == 25 and mult == 1: return "SB"
    if score == 25 and mult == 2: return "DB"
    if mult == 1: return f"S{score}"
    if mult == 2: return f"D{score}"
    if mult == 3: return f"T{score}"
    return f"{mult}x{score}"

# Running "state" from score_change / period_start etc.
state = {
    "home_rem": None,
    "away_rem": None,
    "period": None,        # leg/set index if present
    "leg_home": None,      # legs won (if present)
    "leg_away": None,
    "throwing": None,      # who is currently throwing (optional)
}

# For dart_in_visit
current_thrower = None
dart_in_visit = 0
rows = []

for ev in timeline_events:
    t = ev.get("type")

    # New leg starting
    if t == "period_start":
        state["period"] = ev.get("period") or state["period"]
        current_thrower = None
        dart_in_visit = 0

    # Leg count update (who has how many legs)
    if t == "leg_score_change":
        # keys vary; try common ones
        state["leg_home"] = ev.get("home_score", state["leg_home"])
        state["leg_away"] = ev.get("away_score", state["leg_away"])

    # Remaining score update (THIS is what you want)
    if t == "score_change":
        # In many Sportradar feeds, these are present here (not on dart events)
        if ev.get("home_score") is not None:
            state["home_rem"] = ev.get("home_score")
        if ev.get("away_score") is not None:
            state["away_rem"] = ev.get("away_score")

    # Dart event: record with the latest known state
    if t == "dart":
        comp = ev.get("competitor")  # "home"/"away"

        if comp != current_thrower:
            current_thrower = comp
            dart_in_visit = 0
        dart_in_visit += 1

        rows.append({
            "event_id": ev.get("id"),
            "time": ev.get("time"),
            "competitor": comp,
            "dart_in_visit": dart_in_visit,

            # remaining before dart = current state for that competitor
            "home_remaining": state["home_rem"],
            "away_remaining": state["away_rem"],
            "score_remaining_before": state["home_rem"] if comp == "home" else state["away_rem"],
            "opp_remaining_before": state["away_rem"] if comp == "home" else state["home_rem"],

            "dart_score": ev.get("dart_score"),
            "multiplier": ev.get("dart_score_multiplier"),
            "dart_total": ev.get("dart_score_total"),
            "segment": seg(ev.get("dart_score"), ev.get("dart_score_multiplier")),

            "is_bust": ev.get("is_bust"),
            "is_checkout_attempt": ev.get("is_checkout_attempt"),
            "is_gameshot": ev.get("is_gameshot"),

            "period": state["period"],
            "legs_home": state["leg_home"],
            "legs_away": state["leg_away"],
        })

df_darts_fixed = pd.DataFrame(rows).sort_values(["time","event_id"]).reset_index(drop=True)
df_darts_fixed.head(50)


,event_id,time,competitor,dart_in_visit,home_remaining,away_remaining,score_remaining_before,opp_remaining_before,dart_score,multiplier,dart_total,segment,is_bust,is_checkout_attempt,is_gameshot,period,legs_home,legs_away
0,2263363342,2026-02-06T17:06:17+00:00,home,1,NaN,NaN,NaN,NaN,20,1,20,S20,False,False,None,None,NaN,NaN
1,2263363368,2026-02-06T17:06:20+00:00,home,2,NaN,NaN,NaN,NaN,20,3,60,T20,False,False,None,None,NaN,NaN
2,2263363378,2026-02-06T17:06:21+00:00,home,3,NaN,NaN,NaN,NaN,5,1,5,S5,False,False,None,None,NaN,NaN
3,2263363440,2026-02-06T17:06:24+00:00,away,1,416.0,501.0,501.0,416.0,20,1,20,S20,False,False,None,None,NaN,NaN
4,2263363474,2026-02-06T17:06:26+00:00,away,2,416.0,501.0,501.0,416.0,20,1,20,S20,False,False,None,None,NaN,NaN
5,2263363538,2026-02-06T17:06:29+00:00,away,3,416.0,501.0,501.0,416.0,17,3,51,T17,False,False,None,None,NaN,NaN
6,2263363606,2026-02-06T17:06:33+00:00,home,1,416.0,410.0,416.0,410.0,20,1,20,S20,False,False,None,None,NaN,NaN
7,2263363644,2026-02-06T17:06:35+00:00,home,2,416.0,410.0,416.0,410.0,19,1,19,S19,False,False,None,None,NaN,NaN
8,2263363662,2026-02-06T17:06:36+00:00,home,3,416.0,410.0,416.0,410.0,19,1,19,S19,False,False,None,None,NaN,NaN
9,2263363716,2026-02-06T17:06:40+00:00,away,1,358.0,410.0,410.0,358.0,20,1,20,S20,False,False,None,None,NaN,NaN
